# Replacing, Transforming, and Cleaning Data

The `re.sub` function is the primary tool for replacing patterns in text. This module covers phone number standardization, spelling correction, data formatting, and text cleaning.

In [1]:
import re, csv, io

## `re.sub` Basics

```python
re.sub(pattern, replacement, string, count=0, flags=0)
```
- `pattern` — what to find
- `replacement` — what to replace it with (can reference groups with `\1`, `\2`, …)
- `count` — max number of replacements (0 = all)

In [2]:
# Simple replacement
text = "The color is grey and the neighbour agrees."
text = re.sub(r"grey", "gray", text)
text = re.sub(r"neighbour", "neighbor", text)
print(text)

The color is gray and the neighbor agrees.


## Standardizing Phone Numbers

In [3]:
def standardize_phone(phone):
    """Normalize various phone formats to +1-XXX-XXX-XXXX."""
    # Strip everything except digits
    digits = re.sub(r"\D", "", phone)
    # Format as +1-XXX-XXX-XXXX (assume US numbers with or without country code)
    if len(digits) == 11 and digits.startswith("1"):
        digits = digits[1:]
    if len(digits) == 10:
        return re.sub(r"(\d{3})(\d{3})(\d{4})", r"+1-\1-\2-\3", digits)
    return phone  # return as-is if format is unexpected

phones = [
    "(555) 123-4567",
    "555.123.4567",
    "5551234567",
    "+1 555 123 4567",
    "1-555-123-4567",
]

for p in phones:
    print(f"{p:<25} -> {standardize_phone(p)}")

(555) 123-4567            -> +1-555-123-4567
555.123.4567              -> +1-555-123-4567
5551234567                -> +1-555-123-4567
+1 555 123 4567           -> +1-555-123-4567
1-555-123-4567            -> +1-555-123-4567


## Correcting Misspellings with a Dictionary

In [4]:
MISSPELLINGS = {
    r"\brecieve\b": "receive",
    r"\boccured\b": "occurred",
    r"\bseperate\b": "separate",
    r"\buntill\b":  "until",
    r"\bwierd\b":   "weird",
}

def replace_misspellings(text):
    for pattern, correction in MISSPELLINGS.items():
        text = re.sub(pattern, correction, text, flags=re.IGNORECASE)
    return text

sample = "I wierd thing occured: I did not recieve the package untill Monday."
print(replace_misspellings(sample))

I weird thing occurred: I did not receive the package until Monday.


## Lookbehind and Lookahead Assertions

- **Lookbehind** `(?<=pattern)` — match a position preceded by `pattern`
- **Lookahead** `(?=pattern)` — match a position followed by `pattern`

These are **zero-width** assertions — they don't consume characters.

In [5]:
# Find uppercase letters that are preceded by lowercase AND followed by a digit
text = "aB3cd4eF5GH6"
pattern = r"(?<=[a-z])[A-Z](?=\d)"
matches = re.findall(pattern, text)
print("Matches:", matches)  # ['B', 'F']

Matches: ['B', 'F']


In [6]:
# Use lookahead to split camelCase into words
camel = "myVariableName"
words = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", camel)
print(words)  # 'my Variable Name'

my Variable Name


## Replacing Dates and Prices in a Text File

In [7]:
sample_text = """
Invoice date: 01/15/2024
Due date: 02/15/2024
Total amount: $1,250.00
Discount: $125.00
Final price: $1,125.00
"""

def reformat_date(text):
    """Convert MM/DD/YYYY to YYYY-MM-DD."""
    return re.sub(r"(\d{2})/(\d{2})/(\d{4})", r"\3-\1-\2", text)

def reformat_price(text):
    """Remove commas from dollar amounts."""
    return re.sub(r"\$(\d{1,3}(?:,\d{3})*)(\.\d{2})?",
                  lambda m: "$" + m.group(0)[1:].replace(",", ""), text)

result = reformat_date(sample_text)
result = reformat_price(result)
print(result)


Invoice date: 2024-01-15
Due date: 2024-02-15
Total amount: $1250.00
Discount: $125.00
Final price: $1125.00



## Standardize Text Case and Remove Punctuation

Useful for NLP preprocessing, text analysis, and data cleaning.

In [8]:
def tokenize(text):
    """Remove punctuation, lowercase, and split into tokens."""
    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    # Lowercase
    text = text.lower()
    # Tokenize by whitespace
    return re.split(r"\s+", text.strip())

sentence = "Hello, World! This is Python's regex — isn't it great?"
tokens = tokenize(sentence)
print(tokens)

['hello', 'world', 'this', 'is', 'pythons', 'regex', 'isnt', 'it', 'great']


## Cleaning Data in a CSV File

In [9]:
csv_data = """name,price,category
Widget A,$  12.50,Electronics
Gadget B ,$ 7.99 ,Tools
Item C,$ 100.00,Electronics
"""

def clean_price(price_str):
    """Normalize price: remove $ sign, spaces, keep decimal."""
    cleaned = re.sub(r"[\$\s]", "", price_str)
    return cleaned

def clean_name(name_str):
    """Strip surrounding whitespace."""
    return name_str.strip()

reader = csv.DictReader(io.StringIO(csv_data))
for row in reader:
    row['name']  = clean_name(row['name'])
    row['price'] = clean_price(row['price'])
    print(row)

{'name': 'Widget A', 'price': '12.50', 'category': 'Electronics'}
{'name': 'Gadget B', 'price': '7.99', 'category': 'Tools'}
{'name': 'Item C', 'price': '100.00', 'category': 'Electronics'}


## Handling Unicode and Special Characters

Use `re.UNICODE` (default in Python 3) and `\w`, `\d` etc. to handle international text.

In [10]:
import unicodedata

text = "Héllo Wörld — 你好世界 — café"

# Find all Unicode word characters
words = re.findall(r"\w+", text)
print("Words:", words)

# Remove all non-ASCII characters
ascii_only = re.sub(r"[^\x00-\x7F]", "", text)
print("ASCII only:", ascii_only)

Words: ['Héllo', 'Wörld', '你好世界', 'café']
ASCII only: Hllo Wrld    caf


## Redacting Sensitive Data

In [11]:
REDACT_PATTERNS = [
    (r"\b\d{3}-\d{2}-\d{4}\b",                  "[SSN REDACTED]"),       # Social Security
    (r"\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b", "[CARD REDACTED]"), # Credit card
    (r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", "[EMAIL REDACTED]"), # Email
]

def redact(text):
    for pattern, replacement in REDACT_PATTERNS:
        text = re.sub(pattern, replacement, text)
    return text

sensitive = "Customer alice@example.com with SSN 123-45-6789 paid with card 4111 1111 1111 1111."
print(redact(sensitive))

Customer [EMAIL REDACTED] with SSN [SSN REDACTED] paid with card [CARD REDACTED].
